# Тема 6. Конструирование и отбор признаков

## Практика: новые признаки для «Титаника» — помогут ли?

Сегодня не меняем модель — конструируем новые признаки из того, что уже есть в данных (`Name`, `Cabin`, `SibSp`+`Parch`), и проверяем, действительно ли они поднимают точность.

Самостоятельная работа. Если не хватает метода — см. `lesson06_feature_engineering_theory.ipynb`.

**Это файл с решениями — используйте для самопроверки.**

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42

**1. Базовые признаки (как в темах 3-5) — наша точка отсчёта.**

In [2]:
data = pd.read_csv("../data/titanic_train.csv", index_col="PassengerId")
data["Sex"] = (data["Sex"] == "male").astype(int)
data["Age"] = data["Age"].fillna(data["Age"].median())

base_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
X_base = data[base_features]
y = data["Survived"]

rf_base = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1), X_base, y, cv=5
).mean()
logreg_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
lr_base = cross_val_score(logreg_pipe, X_base, y, cv=5).mean()

print(f"База — лес: {rf_base:.4f}, логрегрессия: {lr_base:.4f}")

База — лес: 0.8137, логрегрессия: 0.7856


**2. Титул из имени.** Колонка `Name` устроена как `"Фамилия, Титул Имя..."` — мы уже доставали титул в практике темы 1 (вопрос про самое популярное имя). Создайте колонку `Title` (`data["Name"].apply(lambda n: n.split(",")[1].split(".")[0].strip())`), объедините редкие титулы (меньше 10 повторений) в категорию `"Rare"`, закодируйте в числа (`.astype("category").cat.codes`).

In [3]:
data["Title"] = data["Name"].apply(lambda n: n.split(",")[1].split(".")[0].strip())
rare_titles = data["Title"].value_counts()[data["Title"].value_counts() < 10].index
data["Title"] = data["Title"].replace(rare_titles, "Rare")
data["Title"] = data["Title"].astype("category").cat.codes
data["Title"].value_counts()

Title
2    517
1    182
3    125
0     40
4     27
Name: count, dtype: int64

**3. Размер семьи и признак "один ли пассажир".** `FamilySize = SibSp + Parch + 1` (сам пассажир), `IsAlone = 1`, если `FamilySize == 1`.

In [4]:
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

**4. Палуба из `Cabin`.** Первая буква `Cabin` — палуба (`"C85"` → `"C"`). У большинства пропуск — считайте их отдельной категорией `"U"` (unknown), а не удаляйте. Закодируйте в числа.

In [5]:
data["Deck"] = data["Cabin"].str[0].fillna("U")
data["Deck"] = data["Deck"].astype("category").cat.codes

**5. Цена на человека.** `FarePerPerson = Fare / FamilySize` — большая семья могла купить один груповой билет, тогда `Fare` завышает цену на конкретного пассажира.

In [6]:
data["FarePerPerson"] = data["Fare"] / data["FamilySize"]

**6. Момент истины.** Соберите новый список признаков (`base_features` + `Title`, `FamilySize`, `IsAlone`, `Deck`, `FarePerPerson`), посчитайте `cross_val_score` для леса и для логрегрессии (`cv=5`), как в шаге 1. Сравните с базой — для какой модели новые признаки помогли больше?

In [7]:
new_features = base_features + ["Title", "FamilySize", "IsAlone", "Deck", "FarePerPerson"]
X_new = data[new_features]

rf_new = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1), X_new, y, cv=5
).mean()
lr_new = cross_val_score(logreg_pipe, X_new, y, cv=5).mean()

print(f"Новые признаки — лес: {rf_new:.4f} (было {rf_base:.4f}), логрегрессия: {lr_new:.4f} (было {lr_base:.4f})")

Новые признаки — лес: 0.8137 (было 0.8137), логрегрессия: 0.7924 (было 0.7856)


_Ожидаемый результат: у леса новые признаки почти не меняют (иногда даже немного ухудшают) точность — лес и так умеет находить похожие взаимодействия сам (например, `Pclass` и `Fare` вместе неявно похожи на `FarePerPerson`). А у логистической регрессии — заметный прирост: линейная модель не видит взаимодействий признаков сама, ей нужно явно подать их в виде готовой колонки. Вывод: конструирование признаков особенно окупается для более простых (линейных) моделей — сложные модели вроде леса уже умеют многое находить сами.

---
## Итог

Следующее занятие — обучение без учителя: посмотрим на тех же клиентов оттока без целевой переменной вообще, найдём естественные сегменты через кластеризацию.